# M7L3 DAP-Seq Analysis Pipeline

```{note}
**Runs on an HPC.** This notebook was written for the Hellbender HPC cluster (University of Missouri) and uses SLURM and/or large sequencing files. It is shown here for reading; saved outputs were removed for privacy. Set `REPO_DIR`, `DATA_DIR` and `WORK_DIR` (see the repository README) and download inputs with `data/fetch_data.sh` before running it on your own cluster.
```

```{admonition} Adapted from
:class: tip
**Source:** Parts of the QC and peak-calling sections are adapted from [Introduction to ChIP-seq using high-performance computing](https://hbctraining.github.io/Intro-to-ChIPseq/), Harvard Chan Bioinformatics Core (HBC), [hbctraining/Intro-to-ChIPseq](https://github.com/hbctraining/Intro-to-ChIPseq) (FastQC, MACS2 and narrowPeak lessons).

**License:** [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

**Changes:** Rewritten for DAP-seq of Arabidopsis ARF transcription factors (GSE60143; O'Malley et al. 2016) with a beads-only control on the Hellbender HPC; the pipeline and remaining text are original.
```


*[Third-party figure omitted from the public version; original source not confirmed.]*

DAP-seq combines high-throughput in vitro TF expression with native genomic DNA and next-generation sequencing. Because TFs are purified directly, the method avoids the need for antibodies or transgenic lines, while the use of real genomic DNA preserves natural sequence context and modifications. When paired with NGS, this allows DAP-seq to generate genome-wide, quantitative TF binding profiles in a scalable and efficient manner.

In this lab we discuss the outline of processing DAP-Seq data once we have performed wet lab experiments.

We will perform the following steps: 

1. **Data Acquisition**
   - Download raw reads from SRA using `fasterq-dump`

2. **Quality Control**
   - Assess read quality with `fastqc`.
   - Trim adapters and filter low-quality reads with `fastp`.
   - Summarize QC reports using `multiqc`.

3. **Read Alignment**
   - Align reads to the reference genome with `bowtie2`.
   - Convert, sort, and index BAM files with `samtools`.
   - Optional: mark duplicates with `picard`.

4. **Peak Calling**
   - Identify binding sites using `macs2`.
   - Process and manipulate intervals with `bedtools` or `bedops`.

5. **Motif Discovery with `HOMER`**
   - Discover enriched motifs with `meme` and `homer`.

6. **Visualization**
   - Generate coverage tracks and heatmaps with `deeptools`.
   - Convert BEDGraph to BigWig for genome browsers using `ucsc-bedgraphtobigwig`.
   - Visualize peaks in IGV (if desired).

7. **Peak Annotation and Gene Assignment**
   - Associate peaks with nearby genes and genomic features

8. **Functional Analysis with R**
   - Perform Gene Ontology (GO) enrichment analysis to understand the biological functions of target genes.
  
A Video lecture on the notebook's content is below:


<iframe width="560" height="315" src="https://www.youtube.com/embed/NyX2dAa0EGE?si=KtDYYm5_KmmAJnNZ" title="YouTube video player" frameborder="0" allow="accelerometer; autoplay; clipboard-write; encrypted-media; gyroscope; picture-in-picture; web-share" referrerpolicy="strict-origin-when-cross-origin" allowfullscreen></iframe>

## 1. Data Acquisition

In this step, we download raw sequencing data from public repositories (e.g., SRA, GEO). In our case, we will use GSE60143.

### Tools
- `fasterq-dump` (SRA Toolkit) - Downloads FASTQ files from SRA
- `pigz` - Parallel gzip compression

### Input
- SRA accession numbers for each sample. Can be fetched using esearch and finding the sample run info (use: `esearch -db gds -query "GSE#####" | efetch -format runinfo > "$RUNINFO"`

### Output
- **FASTQ files** (`.fastq.gz`) - Compressed raw sequencing reads


The FASTQ file format is the defacto file format for sequence reads generated from next-generation sequencing technologies. This file format evolved from FASTA in that it contains sequence data, but also contains quality information. Similar to FASTA, the FASTQ file begins with a header line. The difference is that the FASTQ header is denoted by a @ character. For a single record (sequence read) there are four lines, each of which are described below:

### Example Commands
```bash
# Download ARF5 replicate 1 (GSM1925826)
fasterq-dump SRR2926896 --split-files --threads 8

# Download ARF5 replicate 2 (GSM1925827)
fasterq-dump SRR2926897 --split-files --threads 8

# Download beads-only control (GSM1924998)
fasterq-dump SRR2926068 --split-files --threads 8

# Compress files
pigz -p 8 *.fastq
```

### File Format Review: FASTQ
```
@SRR2926896.1
AGCTTAGCTAGCTAGCTAGCTAGCTAGCTAGCTAGCT
+
IIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIIII
```
- Line 1: Read identifier (@ symbol)
- Line 2: DNA sequence
- Line 3: Separator (+ symbol)
- Line 4: Quality scores (Phred+33 encoding)

## 2. Quality Control

In this step, we assess read quality and remove low-quality sequences and adapter contamination.

### Tools
- `fastqc` - Quality assessment
- `fastp` - Adapter trimming and quality filtering
- `multiqc` - Aggregate QC reports

 FastQC provides a simple way to do some quality control checks on raw sequence data coming from high throughput sequencing pipelines. It provides a modular set of analyses which you can use to give a quick impression of whether your data has any problems of which you should be aware before doing any further analysis.

### Input
- Raw FASTQ files (`.fastq.gz`)

### Output
- **HTML reports** - Quality metrics and plots
- **Trimmed FASTQ files** - Clean reads ready for alignment

### Example Commands
```bash
# Initial quality assessment
fastqc *.fastq.gz -o fastqc_raw/ -t 8

# Trim adapters and filter low-quality reads
fastp -i SRR2926896.fastq.gz \
      -o SRR2926896_trimmed.fastq.gz \
      --length_required 50 \
      --qualified_quality_phred 20 \
      --thread 8

# Aggregate QC reports
multiqc fastqc_raw/ -o multiqc_report/
```
*[Third-party figure omitted from the public version; original source not confirmed.]*

### Resource: [multi-qc](https://biocorecrg.github.io/RNAseq_course_2019/multiqc.html)

### Key Quality Metrics

- Per base sequence composition: You can expect to encounter read start sequence biases in this plot if fragmenting with transposases.
- Sequence duplication levels: This plot can help you get an idea of what proportion of your library corresponds to duplicates. Duplicates are typically removed (even if there is a chance that they are biological duplicates); therefore, if there is a large amount of duplication, the number of reads available for mapping and/or for peak calling will be reduced.
- Over-represented sequences: Over-represented sequences are either highly biologically significant or represent biases. With ChIP-seq you expect to see over-represented sequences in the IP sample because that’s exactly what you’re doing - enriching for particular sequences based on binding affinity. However, lack of over-represented sequences doesn’t mean you have a bad experiment. If you see over-represented sequences in the input, that usually reflects some bias in the protocol to specific regions.



## 3. Read Alignment

We map sequencing reads(segments of DNA the TFs bind to) to the reference genome, in our case, the TAIR10 genome, to determine their position.

*[Third-party figure omitted from the public version; original source not confirmed.]*

### Tools
- `bowtie2` - Short read aligner (used in O'Malley et al. 2016 on GSE60143)
- `samtools` - SAM/BAM manipulation
- `picard` - Mark duplicates (optional)

1. Creating a Bowtie2 index

To perform the Bowtie2 alignment, a genome index is required. The index is analagous to the index in the back of a book. By indexing the genome, we have organized it in a manner that now allows for efficient search and retrieval of matches of the query (sequence read) to the genome. We previously generated the genome indices for you, and they exist in the reference_data directory.

> `bowtie2-build <path_to_reference_genome.fa> <prefix_to_name_indexes>`

2. We will perform alignment on our single raw FASTQ file, H1hesc_Input_Rep1_chr12.fastq. Details on Bowtie2 and its functionality can be found in the user manual; we encourage you to peruse through to get familiar with all available options.

The basic options for aligning reads to the genome using Bowtie2 are:

    -p: number of processors / cores
    -q: reads are in FASTQ format
    --local: local alignment feature to perform soft-clipping
    -x: /path/to/genome_indices_directory
    -U: /path/to/FASTQ_file
    -S: /path/to/output/SAM_file


### Input
- Trimmed FASTQ files
- Reference genome index (TAIR10 for *Arabidopsis*)

### Output
- **SAM files** (`.sam`) - Sequence Alignment/Map format (text)
- **BAM files** (`.bam`) - Binary compressed SAM
- **BAI files** (`.bam.bai`) - BAM index files

  The output we requested from the Bowtie2 aligner is an unsorted SAM file, also known as Sequence Alignment Map format. The SAM file, is a tab-delimited text file that contains information for each individual read and its alignment to the genome. While we will go into some features of the SAM format, the paper by [Heng Li et al](http://bioinformatics.oxfordjournals.org/content/25/16/2078.full) provides a lot more detail on the specification.

### Example Commands
```bash
# Build genome index (one-time step)
bowtie2-build TAIR10.fa TAIR10_index

# Align reads to reference genome
bowtie2 -x TAIR10_index \
        -U SRR2926896_trimmed.fastq.gz \
        -S ARF5_rep1.sam \
        -p 8 \
        --very-sensitive

# Convert SAM to BAM, sort, and index
samtools view -bS ARF5_rep1.sam | samtools sort -o ARF5_rep1.sorted.bam
samtools index ARF5_rep1.sorted.bam

# Optional: Mark duplicates
picard MarkDuplicates \
       I=ARF5_rep1.sorted.bam \
       O=ARF5_rep1.sorted.dedup.bam \
       M=ARF5_rep1.metrics.txt \
       REMOVE_DUPLICATES=false
```

### File Format: SAM/BAM
SAM is tab-delimited with 11 mandatory fields:
```
SRR2926896.1  0  Chr1  12345  60  100M  *  0  0  AGCT...  IIII...
```

The file begins with a header, which is optional. The header is used to describe source of data, reference sequence, method of alignment, etc., this will change depending on the aligner being used. Each section begins with character ‘@’ followed by a two-letter record type code. These are followed by two-letter tags and values. Example of some common sections are provided below:

*[Third-party figure omitted from the public version; original source not confirmed.]*


- Column 1: Read name
- Column 2: Bitwise FLAG
- Column 3: Chromosome
- Column 4: Position
- Column 5: Mapping quality (MAPQ)
- Column 6: CIGAR string

### Filtering reads

An important issue with ChIP-seq data concerns the inclusion of multiple mapped reads (reads mapped to multiple loci on the reference genome). Allowing for multiple mapped reads increases the number of usable reads and the sensitivity of peak detection; however, the number of false positives may also increase [1]. Therefore we need to filter our alignment files to contain only uniquely mapping reads in order to increase confidence in site discovery and improve reproducibility. Since there is no parameter in Bowtie2 to keep only uniquely mapping reads, we will need to perform the following steps to generate alignment files containing only the uniquely mapping reads:

1. Change alignment file format from SAM to BAM

The command we will use is samtools view with the following parameters:

    -h: include header in output
    -S: input is in SAM format
    -b: output BAM format
    -o: /path/to/output/file

3. Sort BAM file by read coordinate locations

Before we can filter to keep the uniquely mapping reads, we need to sort our BAM alignment files by genomic coordinates (instead of by name). To perform this sort, we will use Sambamba, which is a tool that quickly processes BAM and SAM files.

The command we will use is sambamba sort with the following parameters:

    -t: number of threads / cores
    -o: /path/to/output/file

5. Filter to keep only uniquely mapping reads (this will also remove any unmapped reads)

Finally, we can filter the BAM to keep only uniquely mapping reads. We will use the sambamba view command with the following parameters:

    -t: number of threads / cores
    -h: print SAM header before reads
    -f: format of output file (default is SAM)
    -F: set custom filter - we will be using the filter to remove duplicates, multimappers and unmapped reads.

## 4. Peak Calling

We identify peaks within our alignment files: 

*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038).]*

For DAP-seq experiments, what we observe from the alignment files is a strand asymmetry with read densities on the +/- strand, centered around the binding site. The 5’ ends of the selected fragments will form groups on the positive- and negative-strand. The distributions of these groups are then assessed using statistical measures and compared against background (input or mock IP samples) to determine if the site of enrichment is likely to be a real binding site.

### Tools
- `macs2` - Model-based Analysis of ChIP-Seq (also used for DAP-Seq)
- `bedtools` - BED file manipulation
- `bedops` - Efficient genomic interval operations

### MACS2

A commonly used tool for identifying transcription factor binding sites is named Model-based Analysis of ChIP-seq (MACS). The MACS algorithm captures the influence of genome complexity to evaluate the significance of enriched ChIP regions. Although it was developed for the detection of transcription factor binding sites it is also suited for larger regions.

MACS improves the spatial resolution of binding sites through combining the information of both sequencing tag position and orientation. MACS can be easily used either for the DAP sample alone, or along with a control sample which increases specificity of the peak calls. The MACS workflow is depicted below. In this lesson, we will describe the steps in more detail.

*[MACS workflow figure omitted (third-party figure reproduced in the HBC ChIP-seq lessons). See [HBC: Peak calling with MACS2](https://hbctraining.github.io/Intro-to-ChIPseq/lessons/05_peak_calling_macs.html).]*


#### Removing redundancy

MACS provides different options for dealing with duplicate tags at the exact same location, that is tags with the same coordination and the same strand. The default is to keep a single read at each location. The auto option, which is very commonly used, tells MACS to calculate the maximum tags at the exact same location based on binomal distribution using 1e-5 as the pvalue cutoff. An alternative is to set the all option, which keeps every tag. If an integer is specified, then at most that many tags will be kept at the same location. This redundancy is consistently applied for both the ChIP and input samples.

#### MACS2 parameters

There are seven major functions available in MACS2 serving as sub-commands. We will only cover callpeak in this lesson, but you can use macs2 COMMAND -h to find out more, if you are interested.

callpeak is the main function in MACS2 and can be invoked by typing macs2 callpeak. If you type this command without parameters, you will see a full description of commandline options. Here is a shorter list of the commonly used ones:

Input file options

    -t: The IP data file (this is the only REQUIRED parameter for MACS)
    -c: The control or mock data file
    -f: format of input file; Default is “AUTO” which will allow MACS to decide the format automatically.
    -g: mappable genome size which is defined as the genome size which can be sequenced; some precompiled values provided.

Output arguments

    --outdir: MACS2 will save all output files into speficied folder for this option
    -n: The prefix string for output files
    -B/--bdg: store the fragment pileup, control lambda, -log10pvalue and -log10qvalue scores in bedGraph files

Shifting model arguments

    -s: size of sequencing tags. Default, MACS will use the first 10 sequences from your input treatment file to determine it
    --bw: The bandwidth which is used to scan the genome ONLY for model building. Can be set to the expected sonication fragment size.
    --mfold: upper and lower limit for model building

Peak calling arguments

    -q: q-value (minimum FDR) cutoff
    -p: p-value cutoff (instead of q-value cutoff)
    --nolambda: do not consider the local bias/lambda at peak candidate regions
    --broad: broad peak calling

MACS2 Output files
File formats

Before we start exploring the output of MACS2, we’ll briefly talk about the new file formats you will encounter.

narrowPeak:

A narrowPeak (.narrowPeak) file is used by the ENCODE project to provide called peaks of signal enrichment based on pooled, normalized (interpreted) data. It is a BED 6+4 format, which means the first 6 columns of a standard BED file with 4 additional fields:

![narrowPeak.png](images/M7L3_DAP_Seq_Analysis_Pipeline_d5e8a779.png)

*Image: Harvard Chan Bioinformatics Core (HBC), [Intro-to-ChIPseq](https://hbctraining.github.io/Intro-to-ChIPseq/), CC BY 4.0.*

WIG format:

Wiggle format (WIG) allows the display of continuous-valued data in a track format. Wiggle format is line-oriented. It is composed of declaration lines and data lines, and require a separate wiggle track definition line. There are two options for formatting wiggle data: variableStep and fixedStep. These formats were developed to allow the file to be written as compactly as possible.

BedGraph format:

The BedGraph format also allows display of continuous-valued data in track format. This display type is useful for probability scores and transcriptome data. This track type is similar to the wiggle (WIG) format, but unlike the wiggle format, data exported in the bedGraph format are preserved in their original state. For the purposes of visualization, these can be interchangeable.

There should be 6 files output to the results directory for each sample:

    _peaks.narrowPeak: BED6+4 format file which contains the peak locations together with peak summit, pvalue and qvalue
    _peaks.xls: a tabular file which contains information about called peaks. Additional information includes pileup and fold enrichment
    _summits.bed: peak summits locations for every peak. To find the motifs at the binding sites, this file is recommended
    _model.R: an R script which you can use to produce a PDF image about the model based on your data and cross-correlation plot
    _control_lambda.bdg: bedGraph format for input sample
    _treat_pileup.bdg: bedGraph format for treatment sample


## RESOURCE: [Read more about MACS2 here](https://hbctraining.github.io/Intro-to-ChIPseq/lessons/05_peak_calling_macs.html)


### Input
- Sorted BAM files (treatment and control)

### Output
- **Peaks files** (`.narrowPeak`, `.bed`) - Genomic coordinates of binding sites
- **Summits file** (`.summits.bed`) - Peak summit positions
- **BedGraph/Wig files** - Coverage tracks

### Example Commands
```bash
# Call peaks with MACS2 using beads-only control
macs2 callpeak \
      -t ARF5_rep1.sorted.bam \
      -c beads_control.sorted.bam \
      -f BAM \
      -g 1.2e8 \
      -n ARF5_rep1 \
      --outdir peaks/ \
      -q 0.05 \
      --keep-dup all

# Expected output: 3,000-5,000 ARF5 peaks

# Find overlapping peaks between replicates
bedtools intersect \
         -a ARF5_rep1_peaks.narrowPeak \
         -b ARF5_rep2_peaks.narrowPeak \
         -wa > ARF5_reproducible_peaks.bed
```

### File Format: narrowPeak (BED6+4)
```
Chr1  12000  12500  peak_1  1000  .  25.5  100.3  75.2  250
```
- Columns 1-3: Chromosome, start, end
- Column 4: Peak name
- Column 5: Score
- Column 6: Strand
- Column 7: Fold enrichment
- Column 8: -log10(p-value)
- Column 9: -log10(q-value)
- Column 10: Summit position relative to start


## 5. Motif Discovery

We identify DNA sequence motifs enriched in the binding sites using `HOMER`:

*[Figure omitted: O'Malley et al. (2016) Cell 165:1280-1292, [doi:10.1016/j.cell.2016.04.038](https://doi.org/10.1016/j.cell.2016.04.038).]*

### Tools
- `MEME-ChIP` - Comprehensive motif analysis suite
- `FIMO` - Find Individual Motif Occurrences
- `HOMER` - Motif discovery and analysis

### Input
- Peak files (`.bed` or `.narrowPeak`)
- Genome FASTA file

### Output
- **Motif logos** - Visual representation of sequence motifs
- **Position weight matrices (PWM)** - Motif models
- **Motif occurrence files** - Where motifs are found

### Example Commands
```bash
# Extract peak sequences (±100 bp from summit)
bedtools getfasta \
         -fi TAIR10.fa \
         -bed ARF5_peaks.narrowPeak \
         -fo ARF5_peak_sequences.fa

# Run MEME-ChIP for de novo motif discovery
meme-chip \
         -oc ARF5_motifs/ \
         -db JASPAR_PLANTS.meme \
         -meme-maxsize 1000000 \
         ARF5_peak_sequences.fa

# Search for known motifs with FIMO
fimo --oc ARF5_fimo/ \
     --thresh 1e-4 \
     JASPAR_PLANTS.meme \
     ARF5_peak_sequences.fa
```

## 6. Visualization

Use a genome browser (IGV) to observe peaks within the genome for our samples. Files used are in `bed` format, or modifications of `bed` format(`narrowPeak`, `BigWig`, etc).

### Tools
- `deeptools` - Generate coverage tracks, heatmaps, and profiles
- `ucsc-bedgraphtobigwig` - Convert to BigWig format
- `IGV` - Integrative Genomics Viewer

### Input
- BAM files
- Peak files
- Gene annotation files (GTF/GFF)

### Output
- **BigWig files** (`.bw`) - Compressed coverage tracks
- **Heatmaps** (`.png`, `.pdf`) - Signal distribution
- **Profile plots** - Average signal across features

### Example Commands
```bash
# Generate BigWig coverage track
bamCoverage \
    --bam ARF5_rep1.sorted.bam \
    --outFileName ARF5_rep1.bw \
    --binSize 10 \
    --normalizeUsing RPKM \
    --numberOfProcessors 8

# Create heatmap of signal around TSS
computeMatrix reference-point \
    -S ARF5_rep1.bw \
    -R TAIR10_genes.bed \
    --referencePoint TSS \
    -b 2000 -a 2000 \
    -out ARF5_matrix.gz

plotHeatmap \
    -m ARF5_matrix.gz \
    -out ARF5_heatmap.png \
    --colorMap Blues \
    --whatToShow 'heatmap and colorbar'

# Profile plot
plotProfile \
    -m ARF5_matrix.gz \
    -out ARF5_profile.png \
    --perGroup \
    --plotTitle "ARF5 binding around TSS"
```

### Visualization in IGV
1. Load TAIR10 reference genome
2. Load BigWig tracks for both replicates and control
3. Load peak files (`.narrowPeak`)
4. Navigate to example locus: **IAA5 promoter (AT1G15580)**

## 7. Peak Annotation and Gene Assignment

we associate peaks with nearby genes and genomic features.

### Tools
- `ChIPseeker` (R/Bioconductor)
- `HOMER annotatePeaks.pl`
- `bedtools closest`

### Example Commands
```bash
# Annotate peaks with HOMER
annotatePeaks.pl \
    ARF5_peaks.narrowPeak \
    TAIR10 \
    -gtf TAIR10_genes.gtf \
    > ARF5_peaks_annotated.txt
```

### Expected Genomic Distribution
- **Promoter regions** (±1 kb from TSS): ~40-50%
- **Intergenic**: ~30-40%
- **Introns**: ~10-20%
- **Exons**: ~5-10%

## 8. Functional Analysis with R

we will perform Gene Ontology (GO) enrichment analysis to understand the biological functions of target genes. In our case, ARF5.

### R Packages Required
```r
# Install packages (run once)
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")

BiocManager::install(c(
    "clusterProfiler",
    "org.At.tair.db",
    "enrichplot",
    "ggplot2",
    "ChIPseeker",
    "TxDb.Athaliana.BioMart.plantsmart28"
))
```

### Step 1: Load Libraries and Data
```r
# Load required libraries
library(clusterProfiler)
library(org.At.tair.db)
library(enrichplot)
library(ggplot2)
library(ChIPseeker)
library(TxDb.Athaliana.BioMart.plantsmart28)

# Read peak file
peaks <- read.table("ARF5_reproducible_peaks.bed", 
                    header = FALSE, 
                    stringsAsFactors = FALSE)
colnames(peaks) <- c("chr", "start", "end", "name", "score")

# Convert to GRanges object
library(GenomicRanges)
peaks_gr <- GRanges(
    seqnames = peaks$chr,
    ranges = IRanges(start = peaks$start, end = peaks$end)
)
```

### Step 2: Annotate Peaks to Genes
```r
# Load Arabidopsis annotation
txdb <- TxDb.Athaliana.BioMart.plantsmart28

# Annotate peaks
peak_anno <- annotatePeak(
    peaks_gr,
    tssRegion = c(-1000, 1000),
    TxDb = txdb,
    annoDb = "org.At.tair.db"
)

# View annotation summary
plotAnnoPie(peak_anno)

# Extract gene list
genes <- as.data.frame(peak_anno)$geneId
genes <- unique(genes)

# Convert to AGI locus IDs (remove transcript variants)
genes <- sapply(strsplit(genes, "\\."), function(x) x[1])
genes <- unique(genes)

cat("Number of ARF5 target genes:", length(genes), "\n")
```

### Step 3: Gene Ontology Enrichment Analysis
```r
# GO Biological Process enrichment
ego_bp <- enrichGO(
    gene = genes,
    OrgDb = org.At.tair.db,
    keyType = "TAIR",
    ont = "BP",
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05,
    readable = TRUE
)

# View top results
head(ego_bp, 20)

# Visualize results
dotplot(ego_bp, showCategory = 20) +
    ggtitle("ARF5 Target Genes - GO Biological Process")

# Bar plot
barplot(ego_bp, showCategory = 20) +
    ggtitle("ARF5 GO Enrichment (Biological Process)")

# GO Molecular Function enrichment
ego_mf <- enrichGO(
    gene = genes,
    OrgDb = org.At.tair.db,
    keyType = "TAIR",
    ont = "MF",
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05,
    readable = TRUE
)

# GO Cellular Component enrichment
ego_cc <- enrichGO(
    gene = genes,
    OrgDb = org.At.tair.db,
    keyType = "TAIR",
    ont = "CC",
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05,
    readable = TRUE
)
```

### Step 4: Visualize Enrichment Results
```r
# Gene-Concept Network
cnetplot(ego_bp, 
         categorySize = "pvalue",
         colorEdge = TRUE,
         showCategory = 10) +
    ggtitle("Gene-Concept Network for ARF5 Targets")

# Enrichment Map (shows relationship between terms)
emapplot(pairwise_termsim(ego_bp), 
         showCategory = 30,
         cex_label_category = 0.7)

# Heatmap plot
heatplot(ego_bp, showCategory = 20)

# UpSet plot (shows gene overlap between categories)
upsetplot(ego_bp, n = 15)
```

### Step 5: KEGG Pathway Enrichment
```r
# KEGG pathway enrichment
# Note: Use gene IDs compatible with KEGG
# For Arabidopsis, we need to convert to KEGG IDs

# Get KEGG IDs
gene_entrez <- bitr(
    genes,
    fromType = "TAIR",
    toType = "ENTREZID",
    OrgDb = org.At.tair.db
)

kegg <- enrichKEGG(
    gene = gene_entrez$ENTREZID,
    organism = "ath",  # Arabidopsis thaliana
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05
)

# Visualize KEGG results
dotplot(kegg, showCategory = 20) +
    ggtitle("ARF5 Target Genes - KEGG Pathways")
```

### Step 6: Compare with Background
```r
# Get all genes in genome as background
all_genes <- keys(org.At.tair.db, keytype = "TAIR")

# Perform enrichment with background
ego_with_bg <- enrichGO(
    gene = genes,
    universe = all_genes,
    OrgDb = org.At.tair.db,
    keyType = "TAIR",
    ont = "BP",
    pAdjustMethod = "BH",
    pvalueCutoff = 0.05,
    qvalueCutoff = 0.05,
    readable = TRUE
)
```

### Step 7: Export Results
```r
# Save results as CSV
write.csv(as.data.frame(ego_bp), 
          file = "ARF5_GO_BP_enrichment.csv",
          row.names = FALSE)

write.csv(as.data.frame(ego_mf),
          file = "ARF5_GO_MF_enrichment.csv",
          row.names = FALSE)

write.csv(as.data.frame(kegg),
          file = "ARF5_KEGG_enrichment.csv",
          row.names = FALSE)

# Save plots
pdf("ARF5_functional_analysis.pdf", width = 12, height = 10)
dotplot(ego_bp, showCategory = 30)
barplot(ego_bp, showCategory = 20)
cnetplot(ego_bp, categorySize = "pvalue", showCategory = 10)
emapplot(pairwise_termsim(ego_bp), showCategory = 30)
dev.off()
```

### Expected Results for ARF5 (MONOPTEROS)

Based on O'Malley et al. 2016, ARF5 target genes should show significant enrichment for:

**Biological Processes** (p < 10⁻⁶⁰):
- Auxin-activated signaling pathway
- Response to auxin
- Embryo development
- Root development
- Cell differentiation
- Tissue development
- Plant organ development

**Molecular Functions**:
- DNA-binding transcription factor activity
- Auxin response element binding
- Protein dimerization activity

**Example Code for Auxin-Specific Analysis**:
```r
# Filter for auxin-related terms
auxin_terms <- ego_bp[grep("auxin", ego_bp$Description, ignore.case = TRUE), ]
print(auxin_terms)

# Create custom plot for auxin pathways
ggplot(as.data.frame(auxin_terms), 
       aes(x = reorder(Description, -log10(qvalue)), 
           y = -log10(qvalue))) +
    geom_col(fill = "steelblue") +
    coord_flip() +
    theme_minimal() +
    labs(title = "ARF5 Enrichment in Auxin-Related Processes",
         x = "GO Term",
         y = "-log10(q-value)") +
    geom_hline(yintercept = -log10(0.05), 
               linetype = "dashed", 
               color = "red")
```

### Step 8: Gene Set Enrichment Analysis (GSEA)
```r
# If you have gene expression data or peak scores
# Create a ranked gene list

# Example: rank by peak score or fold enrichment
gene_scores <- setNames(peaks_annotated$fold_enrichment, 
                        peaks_annotated$geneId)
gene_scores <- sort(gene_scores, decreasing = TRUE)

# Run GSEA
gsea_result <- gseGO(
    geneList = gene_scores,
    OrgDb = org.At.tair.db,
    keyType = "TAIR",
    ont = "BP",
    pvalueCutoff = 0.05,
    pAdjustMethod = "BH"
)

# Visualize GSEA results
gseaplot2(gsea_result, 
          geneSetID = 1:5,
          title = "GSEA Plot - Top 5 Pathways")
```

### Interpretation Tips

1. **P-value vs Q-value**: 
   - P-value: statistical significance
   - Q-value: corrected for multiple testing (use this!)

2. **Gene Ratio**: 
   - Number of target genes / Total genes in category
   - Higher ratio = stronger enrichment

3. **Rich Factor**:
   - Gene Ratio / Background Ratio
   - Values > 1 indicate enrichment

4. **Expected Results**:
   - ARF5 should show extreme enrichment (p < 10⁻⁶⁰) for auxin response
   - Embryonic and root development terms should be prominent
   - This validates ARF5's known role as MONOPTEROS

# Resources
- [MACS2 documentation](https://github.com/macs3-project/MACS)
- [deepTools documentation](https://deeptools.readthedocs.io/)
- [MEME Suite tutorials](http://meme-suite.org/doc/overview.html)
- [Bioconductor clusterProfiler](https://bioconductor.org/packages/release/bioc/html/clusterProfiler.html)
- [TAIR Arabidopsis database](https://www.arabidopsis.org/)